# Dimensionality Reduction — Guided Notebook

Explore PCA, explained variance, t-SNE, TruncatedSVD, random projection, and downstream model evaluation. Synthetic datasets are generated locally.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_blobs, make_swiss_roll
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.random_projection import GaussianRandomProjection
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
RANDOM_STATE=42

## 2. Create correlated high-dimensional data

In [ ]:
X, y = make_classification(n_samples=900, n_features=24, n_informative=7,
                           n_redundant=10, n_repeated=0, random_state=RANDOM_STATE)
print("Shape:", X.shape)
corr = pd.DataFrame(X).corr()
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.title("Feature correlation matrix")
plt.show()

## 3. Split and standardize without leakage

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. PCA explained variance

In [ ]:
pca_all = PCA().fit(X_train_scaled)
cumulative = np.cumsum(pca_all.explained_variance_ratio_)
plt.plot(np.arange(1, len(cumulative)+1), cumulative, marker="o")
plt.axhline(0.90, linestyle="--", label="90% variance")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.legend()
plt.title("PCA cumulative explained variance")
plt.show()
n90 = int(np.searchsorted(cumulative, 0.90) + 1)
print("Components for at least 90% variance:", n90)

## 5. Project into two PCA dimensions

In [ ]:
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca2.fit_transform(X_train_scaled)
plt.scatter(X_2d[:,0], X_2d[:,1], c=y_train, s=12, alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA 2D projection (labels used only for coloring)")
plt.show()
print("Variance explained by first two PCs:", pca2.explained_variance_ratio_.sum())

## 6. t-SNE visualization

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(X_train_scaled), size=min(450, len(X_train_scaled)), replace=False)
X_tsne_input = X_train_scaled[idx]
y_tsne = y_train[idx]
tsne = TSNE(n_components=2, perplexity=30, init="pca",
            learning_rate="auto", max_iter=750, random_state=RANDOM_STATE)
embedding = tsne.fit_transform(X_tsne_input)
plt.scatter(embedding[:,0], embedding[:,1], c=y_tsne, s=12, alpha=0.7)
plt.title("t-SNE exploratory view")
plt.show()

t-SNE is mainly for local-neighborhood visualization. Do not interpret distances between far-apart islands as faithful global distances or infer clusters from appearance alone.

## 7. TruncatedSVD and random projection

In [ ]:
from scipy.sparse import csr_matrix
rng = np.random.default_rng(RANDOM_STATE)
sparse_data = csr_matrix(rng.poisson(0.12, size=(300,100)))
svd = TruncatedSVD(n_components=10, random_state=RANDOM_STATE)
X_svd = svd.fit_transform(sparse_data)
rp = GaussianRandomProjection(n_components=10, random_state=RANDOM_STATE)
X_rp = rp.fit_transform(sparse_data)
print("Sparse input:", sparse_data.shape)
print("TruncatedSVD output:", X_svd.shape)
print("Random projection output:", X_rp.shape)
print("SVD explained variance sum:", round(svd.explained_variance_ratio_.sum(), 3))

## 8. Compare downstream classification with and without PCA

In [ ]:
no_pca = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])
with_pca = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.90, random_state=RANDOM_STATE)),
    ("classifier", LogisticRegression(max_iter=2000))
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for name, estimator in [("No PCA", no_pca), ("PCA 90%", with_pca)]:
    scores = cross_validate(estimator, X_train, y_train, cv=cv,
                            scoring=["accuracy", "f1", "roc_auc"])
    print(name, {metric: round(scores[f"test_{metric}"].mean(), 3)
                 for metric in ["accuracy", "f1", "roc_auc"]})

## 9. Fit selected pipeline and evaluate held-out test set

In [ ]:
with_pca.fit(X_train, y_train)
pred = with_pca.predict(X_test)
proba = with_pca.predict_proba(X_test)[:,1]
print(classification_report(y_test, pred))
print("Test ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print("PCA components:", with_pca.named_steps["pca"].n_components_)

## 10. Optional: Swiss roll manifold

In [ ]:
X_roll, color = make_swiss_roll(n_samples=700, noise=0.08, random_state=RANDOM_STATE)
roll_pca = PCA(n_components=2).fit_transform(X_roll)
roll_tsne = TSNE(n_components=2, init="pca", learning_rate="auto",
                 perplexity=30, max_iter=750, random_state=RANDOM_STATE).fit_transform(X_roll)
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].scatter(roll_pca[:,0], roll_pca[:,1], c=color, s=8)
axes[0].set_title("Swiss roll: PCA")
axes[1].scatter(roll_tsne[:,0], roll_tsne[:,1], c=color, s=8)
axes[1].set_title("Swiss roll: t-SNE")
plt.tight_layout()
plt.show()

## 11. Conclusions and extension

- Try a few PCA component counts using training cross-validation.
- Compare performance and runtime against the pipeline without PCA.
- Repeat t-SNE with another seed or perplexity and describe what changes.
- Document what the 2D plots do and do not establish.

**Key point:** dimensionality reduction is not automatically better. Choose it based on the task, evaluation, and acceptable loss of information.